# Project 3: Grouping Data into Clusters

Welcome to the final boss! In this project, we will solve the problem of automatically grouping data into distinct clusters without knowing the groups in advance.

**Algorithm Context:** To solve this, we will build a complete **K-Means Clustering** machine learning algorithm entirely from scratch using only NumPy.

K-Means is an unsupervised algorithm that groups data into $K$ distinct clusters. 
The algorithm works in 3 iterative steps:
1. **Initialize**: Randomly choose $K$ points as the starting cluster centers (centroids).
2. **Assign**: Assign every data point to its closest centroid (using Euclidean distance).
3. **Update**: Move each centroid to the mean (average) position of all points assigned to it.
4. **Repeat** steps 2 and 3 until the centroids stop moving.

This project will test your ultimate mastery of NumPy, particularly:
- Advanced Broadcasting (`np.newaxis`)
- Aggregation along specific axes
- Boolean Indexing and Array Manipulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed
np.random.seed(42)

# Generate 3 "blobs" of data (3 distinct clusters)
cluster_1 = np.random.normal(loc=[20, 20], scale=5, size=(100, 2))
cluster_2 = np.random.normal(loc=[60, 80], scale=6, size=(100, 2))
cluster_3 = np.random.normal(loc=[80, 20], scale=5, size=(100, 2))

# Combine them into a single dataset of 300 points
data = np.vstack([cluster_1, cluster_2, cluster_3])

# Plot the raw, unclustered data
plt.figure(figsize=(8, 6))
plt.scatter(data[:, 0], data[:, 1], color='gray', alpha=0.6)
plt.title("Raw Unclustered Data")
plt.show()

## 1. Initialize Centroids
First, we need to pick $K=3$ random points from our data to act as our initial centroids.

In [ ]:
K = 3

# Randomly select 3 unique indices from the data
random_indices = np.random.choice(data.shape[0], size=K, replace=False)

# Get the coordinates of these initial centroids
centroids = data[random_indices]

print("Initial Centroids:\n", centroids)

## 2. The Assignment Step (Broadcasting Magic)
We need to calculate the distance from EVERY point (300 points) to EVERY centroid (3 points). 

To do this without slow `for` loops, we use advanced broadcasting. 
- `data` has shape `(300, 2)`
- `centroids` has shape `(3, 2)`

If we add a new axis to `data` to make it `(300, 1, 2)`, NumPy will automatically broadcast `centroids` across that new axis, allowing us to calculate all 900 distances instantly!

In [ ]:
# 1. Expand dimensions for broadcasting
# data[:, np.newaxis] changes shape to (300, 1, 2)
# centroids changes shape to (3, 2) -> broadcasts to (1, 3, 2) internally
differences = data[:, np.newaxis] - centroids

# 2. Calculate Euclidean distance (sum of squared differences along the coordinate axis)
# axis=2 is the coordinate axis (X, Y)
distances = np.sqrt(np.sum(differences**2, axis=2))

# Now distances has shape (300, 3). For each point, we have 3 distances!
print("Distances shape:", distances.shape)

# 3. Assign each point to the closest centroid using np.argmin
# We look across the 3 centroids (axis=1) to find the minimum distance index
cluster_assignments = np.argmin(distances, axis=1)

print("First 10 assignments:", cluster_assignments[:10])

## 3. The Update Step
Now that every point belongs to cluster 0, 1, or 2, we need to move the centroids to the **mean** of their respective points.

In [ ]:
new_centroids = np.zeros_like(centroids)

for i in range(K):
    # Find all points assigned to cluster i using boolean indexing
    points_in_cluster = data[cluster_assignments == i]
    
    # Calculate the mean of these points and assign it as the new centroid
    new_centroids[i] = np.mean(points_in_cluster, axis=0)

print("Old Centroids:\n", centroids)
print("\nNew Centroids:\n", new_centroids)

## 4. Putting it all together
Let's put the Assignment and Update steps inside a loop and run it until the centroids stop changing (convergence)!

In [ ]:
# Re-initialize for the full run
centroids = data[np.random.choice(data.shape[0], size=K, replace=False)]
max_iterations = 10

for iteration in range(max_iterations):
    # 1. Assignment Step
    distances = np.sqrt(np.sum((data[:, np.newaxis] - centroids)**2, axis=2))
    cluster_assignments = np.argmin(distances, axis=1)
    
    # 2. Update Step
    new_centroids = np.zeros_like(centroids)
    for i in range(K):
        # Handle edge case: if a cluster loses all points, keep it where it is
        if np.any(cluster_assignments == i):
            new_centroids[i] = np.mean(data[cluster_assignments == i], axis=0)
        else:
            new_centroids[i] = centroids[i]
            
    # 3. Check for convergence (if centroids didn't move, we're done!)
    if np.allclose(centroids, new_centroids):
        print(f"Converged after {iteration + 1} iterations!")
        break
        
    centroids = new_centroids

print("Final Centroids:\n", centroids)

## 5. Final Visualization
Let's see our perfectly clustered data!

In [ ]:
plt.figure(figsize=(10, 8))

# Plot each cluster with a different color
colors = ['red', 'blue', 'green']
for i in range(K):
    points = data[cluster_assignments == i]
    plt.scatter(points[:, 0], points[:, 1], color=colors[i], alpha=0.5, label=f'Cluster {i}')

# Plot the final centroids as black stars
plt.scatter(centroids[:, 0], centroids[:, 1], color='black', marker='*', s=300, label='Centroids', edgecolor='white')

plt.title("K-Means Clustering Results")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()